In [0]:
from pyspark.sql.functions import col, avg, count, round

# Solo notas válidas
df_notas = spark.table("main.edu_lakehouse.silver_notas").filter(col("estado_nota") == "valida")
df_materias = spark.table("main.edu_lakehouse.silver_materias")

gold_promedio_materia = (
    df_notas
    .groupBy("materia_id")
    .agg(
        round(avg("nota"), 2).alias("promedio_nota"),
        count("alumno_id").alias("cantidad_examenes")
    )
    .join(df_materias.select("materia_id", "nombre"), "materia_id")
    .orderBy("promedio_nota")
)

gold_promedio_materia.show(truncate=False)

(gold_promedio_materia.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.gold_promedio_materia"))
print("✓ gold_promedio_materia creada")

In [0]:
from pyspark.sql.functions import col, avg, round, count

df_asistencia = spark.table("main.edu_lakehouse.silver_asistencia")
df_alumnos = spark.table("main.edu_lakehouse.silver_alumnos")

gold_asistencia_alumno = (
    df_asistencia
    .groupBy("alumno_id")
    .agg(
        round(avg(col("presente")) * 100, 1).alias("pct_asistencia"),
        count("fecha").alias("clases_totales")
    )
    .join(df_alumnos.select("alumno_id", "nombre", "carrera"), "alumno_id")
    .orderBy("pct_asistencia")
)

gold_asistencia_alumno.show(truncate=False)

(gold_asistencia_alumno.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.gold_asistencia_alumno"))
print("✓ gold_asistencia_alumno creada")

In [0]:
from pyspark.sql.functions import col, avg, round, when, lit

df_notas = spark.table("main.edu_lakehouse.silver_notas").filter(col("estado_nota") == "valida")
df_asistencia = spark.table("main.edu_lakehouse.silver_asistencia")
df_alumnos = spark.table("main.edu_lakehouse.silver_alumnos")

promedios = (
    df_notas.groupBy("alumno_id")
    .agg(round(avg("nota"), 2).alias("promedio_nota"))
)

asistencias = (
    df_asistencia.groupBy("alumno_id")
    .agg(round(avg(col("presente")) * 100, 1).alias("pct_asistencia"))
)

gold_riesgo = (
    df_alumnos
    .join(promedios, "alumno_id", "left")
    .join(asistencias, "alumno_id", "left")
    .withColumn(
        "riesgo",
        when((col("promedio_nota") < 4) | (col("pct_asistencia") < 60), "EN RIESGO")
        .when(col("promedio_nota").isNull(), "SIN NOTA")
        .otherwise("REGULAR")
    )
    .orderBy("riesgo", "promedio_nota")
    .select("alumno_id", "nombre", "carrera", "promedio_nota", "pct_asistencia", "riesgo")
)

gold_riesgo.show(truncate=False)

(gold_riesgo.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.gold_riesgo_alumnos"))
print("✓ gold_riesgo_alumnos creada")

In [0]:
print("=== TABLAS GOLD ===")
spark.sql("SHOW TABLES IN main.edu_lakehouse").\
    filter(col("tableName").startswith("gold")).show(truncate=False)

print("\n=== RESUMEN EJECUTIVO ===")
total = spark.table("main.edu_lakehouse.silver_alumnos").count()
riesgo = spark.table("main.edu_lakehouse.gold_riesgo_alumnos")\
    .filter(col("riesgo") == "EN RIESGO").count()
sin_nota = spark.table("main.edu_lakehouse.gold_riesgo_alumnos")\
    .filter(col("riesgo") == "SIN NOTA").count()

print(f"Total alumnos: {total}")
print(f"En riesgo:     {riesgo}")
print(f"Sin nota:      {sin_nota}")

## Gobierno de datos — esquema de permisos y acceso
La clase 6 establece que el gobierno define quién puede ver cada dato, quién puede
modificar tablas, qué datos son sensibles y qué trazabilidad queda registrada.

In [0]:
print("=== Esquema de gobierno de datos ===")

gobierno = spark.createDataFrame([
    ("Docente",    "silver_notas",          "LECTURA",     "Ve notas individuales de sus materias únicamente"),
    ("Docente",    "silver_asistencia",     "LECTURA",     "Ve asistencia de sus materias únicamente"),
    ("Preceptor",  "silver_asistencia",     "LECTURA",     "Ve asistencia de todos los alumnos"),
    ("Preceptor",  "gold_asistencia_alumno","LECTURA",     "Ve porcentaje de asistencia agregado"),
    ("Directivo",  "gold_promedio_materia", "LECTURA",     "Ve promedios agregados por materia"),
    ("Directivo",  "gold_riesgo_alumnos",   "LECTURA",     "Ve lista de alumnos en riesgo"),
    ("Directivo",  "gold_kpis_calidad",     "LECTURA",     "Ve tablero de calidad de datos"),
    ("Ing. Datos", "bronze_*",              "LECTURA",     "Acceso completo para auditoría y trazabilidad"),
    ("Ing. Datos", "silver_*",              "LECTURA/ESCRITURA", "Administra transformaciones y cuarentena"),
    ("Ing. Datos", "edu_quarantine.*",      "LECTURA/ESCRITURA", "Revisa y gestiona registros rechazados"),
], ["rol", "tabla", "permiso", "descripcion"])

gobierno.show(truncate=False)

print("\n=== Datos sensibles identificados ===")
sensibles = spark.createDataFrame([
    ("silver_notas",      "nota",         "SENSIBLE", "Dato académico individual — acceso restringido por rol"),
    ("silver_alumnos",    "nombre",       "SENSIBLE", "Dato personal — sujeto a normativa de privacidad"),
    ("silver_asistencia", "presente",     "SENSIBLE", "Dato académico individual — acceso restringido por rol"),
    ("gold_riesgo_alumnos","riesgo",      "SENSIBLE", "Clasificación académica — solo directivos y tutores"),
    ("gold_kpis_calidad", "valor",        "INTERNO",  "Indicadores de calidad — uso interno del área de datos"),
], ["tabla", "campo", "clasificacion", "motivo"])

sensibles.show(truncate=False)

print("\n=== Tablas de cuarentena disponibles para auditoría ===")
spark.sql("SHOW TABLES IN main.edu_quarantine").show(truncate=False)